In [1]:
from pyspark.sql import functions as F
from pathlib import Path
import pandas as pd
import math
import json

SILVER_TABLE = "lh_airops_silver.slv_flights_weather_enriched"

FACT_TABLE = "fact_flight_performance"
AGG_TABLE = "agg_daily_origin_airport_performance"
KPI_TABLE = "gold_kpi_snapshot"

BUILD_NOTEBOOK = "nb_gold_mvp_flight_fact_daily_airport"
KPI_NOTEBOOK = "nb_gold_dimensions_kpis"


def stable_gold_signature():
    fact = spark.table(FACT_TABLE)
    agg = spark.table(AGG_TABLE)
    kpi = spark.table(KPI_TABLE).first().asDict()

    fact_rows = fact.count()
    fact_distinct_keys = (
        fact.select("flight_key")
            .distinct()
            .count()
    )

    fact_duplicate_groups = (
        fact.groupBy("flight_key")
            .count()
            .filter(F.col("count") > 1)
            .count()
    )

    aggregate_rows = agg.count()

    aggregate_duplicate_groups = (
        agg.groupBy(
            "flight_date",
            "origin_airport_code"
        )
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    return {
        "fact_rows": fact_rows,
        "fact_distinct_flight_keys": fact_distinct_keys,
        "fact_duplicate_groups": fact_duplicate_groups,
        "aggregate_rows": aggregate_rows,
        "aggregate_duplicate_groups": aggregate_duplicate_groups,
        "kpi_total_flights": int(kpi["total_flights"]),
        "kpi_cancelled_flights": int(kpi["cancelled_flights"]),
        "kpi_arrival_eligible": int(
            kpi["arrival_delay_eligible_flights"]
        ),
        "kpi_arrival_delayed_15": int(
            kpi["arrival_delayed_15_flights"]
        ),
    }


before = stable_gold_signature()

assert (
    before["fact_rows"]
    == before["fact_distinct_flight_keys"]
)
assert before["fact_duplicate_groups"] == 0
assert before["aggregate_duplicate_groups"] == 0

print("=" * 72)
print("GOLD STATE BEFORE SAME-BATCH RERUN")
print("=" * 72)

for key, value in before.items():
    print(f"{key:<34} {value:,}")

print()
print("PRE-RERUN GRAIN CHECK: PASS")

StatementMeta(, a1f909f2-5b97-4107-9d67-41168b0bd666, 3, Finished, Available, Finished, False)

GOLD STATE BEFORE SAME-BATCH RERUN
fact_rows                          597,919
fact_distinct_flight_keys          597,919
fact_duplicate_groups              0
aggregate_rows                     10,031
aggregate_duplicate_groups         0
kpi_total_flights                  597,919
kpi_cancelled_flights              5,402
kpi_arrival_eligible               591,206
kpi_arrival_delayed_15             119,417

PRE-RERUN GRAIN CHECK: PASS


In [2]:
print("=" * 72)
print("SAME-BATCH GOLD RERUN")
print("=" * 72)

print("1/2 Rebuilding Gold fact + daily aggregate...")

notebookutils.notebook.run(
    BUILD_NOTEBOOK,
    600
)

print("2/2 Rebuilding dimensions + KPI snapshot...")

notebookutils.notebook.run(
    KPI_NOTEBOOK,
    600
)

after = stable_gold_signature()

assert before == after, (
    "STOP: Stable Gold business signature changed "
    "after same-batch rerun."
)

assert after["fact_duplicate_groups"] == 0
assert after["aggregate_duplicate_groups"] == 0

print()
print("=" * 72)
print("SAME-BATCH RERUN RESULT")
print("=" * 72)

for key, value in after.items():
    print(f"{key:<34} {value:,}")

print()
print("IDEMPOTENCY GATE: PASS")
print("Same input did not create duplicate business rows.")

StatementMeta(, a1f909f2-5b97-4107-9d67-41168b0bd666, 4, Finished, Available, Finished, False)

SAME-BATCH GOLD RERUN
1/2 Rebuilding Gold fact + daily aggregate...


2/2 Rebuilding dimensions + KPI snapshot...



SAME-BATCH RERUN RESULT
fact_rows                          597,919
fact_distinct_flight_keys          597,919
fact_duplicate_groups              0
aggregate_rows                     10,031
aggregate_duplicate_groups         0
kpi_total_flights                  597,919
kpi_cancelled_flights              5,402
kpi_arrival_eligible               591,206
kpi_arrival_delayed_15             119,417

IDEMPOTENCY GATE: PASS
Same input did not create duplicate business rows.


In [3]:
silver = spark.table(SILVER_TABLE)
gold_kpi = spark.table(KPI_TABLE).first().asDict()

silver_rows = silver.count()

silver_distinct_keys = (
    silver.select("flight_key")
          .distinct()
          .count()
)

assert silver_rows == silver_distinct_keys


silver_kpi = (
    silver
    .agg(
        F.count("*")
         .alias("total_flights"),

        F.sum(
            F.when(
                F.col("cancelled") == True,
                1
            ).otherwise(0)
        ).alias("cancelled_flights"),

        F.sum(
            F.when(
                F.col("arr_del15").isNotNull(),
                1
            ).otherwise(0)
        ).alias("arrival_delay_eligible_flights"),

        F.sum(
            F.when(
                F.col("arr_del15") == True,
                1
            ).otherwise(0)
        ).alias("arrival_delayed_15_flights"),
    )
    .first()
    .asDict()
)


silver_cancellation_rate = (
    silver_kpi["cancelled_flights"]
    / silver_kpi["total_flights"]
)

silver_arrival_delay_rate = (
    silver_kpi["arrival_delayed_15_flights"]
    / silver_kpi["arrival_delay_eligible_flights"]
)


gold_cancellation_rate = float(
    gold_kpi["cancellation_rate"]
)

gold_arrival_delay_rate = float(
    gold_kpi["arrival_delay_15_rate"]
)


assert (
    int(gold_kpi["total_flights"])
    == silver_kpi["total_flights"]
)

assert (
    int(gold_kpi["cancelled_flights"])
    == silver_kpi["cancelled_flights"]
)

assert (
    int(gold_kpi["arrival_delay_eligible_flights"])
    == silver_kpi["arrival_delay_eligible_flights"]
)

assert (
    int(gold_kpi["arrival_delayed_15_flights"])
    == silver_kpi["arrival_delayed_15_flights"]
)

assert math.isclose(
    gold_cancellation_rate,
    silver_cancellation_rate,
    rel_tol=0,
    abs_tol=1e-12
)

assert math.isclose(
    gold_arrival_delay_rate,
    silver_arrival_delay_rate,
    rel_tol=0,
    abs_tol=1e-12
)


release_rows = [
    {
        "metric": "Total Flights",
        "gold_value": f"{int(gold_kpi['total_flights']):,}",
        "silver_value": f"{silver_kpi['total_flights']:,}",
        "definition": "All accepted scheduled flight rows",
        "status": "PASS",
    },
    {
        "metric": "Cancellation Rate",
        "gold_value": f"{gold_cancellation_rate:.4%}",
        "silver_value": f"{silver_cancellation_rate:.4%}",
        "definition": (
            f"{silver_kpi['cancelled_flights']:,} cancelled / "
            f"{silver_kpi['total_flights']:,} scheduled flights"
        ),
        "status": "PASS",
    },
    {
        "metric": "Arrival Delay >=15 Rate",
        "gold_value": f"{gold_arrival_delay_rate:.4%}",
        "silver_value": f"{silver_arrival_delay_rate:.4%}",
        "definition": (
            f"{silver_kpi['arrival_delayed_15_flights']:,} delayed / "
            f"{silver_kpi['arrival_delay_eligible_flights']:,} "
            "known arrival outcomes"
        ),
        "status": "PASS",
    },
]

release_pdf = pd.DataFrame(release_rows)

print("=" * 72)
print("GOLD -> SILVER KPI RECONCILIATION")
print("=" * 72)

print(f"Silver flight rows:                {silver_rows:,}")
print(f"Silver distinct flight_key:        {silver_distinct_keys:,}")
print()
print(f"Total flights:                     {silver_kpi['total_flights']:,}")
print(f"Cancelled flights:                 {silver_kpi['cancelled_flights']:,}")
print(f"Cancellation rate:                 {silver_cancellation_rate:.4%}")
print()
print(
    "Arrival delay >=15 flights:       "
    f"{silver_kpi['arrival_delayed_15_flights']:,}"
)
print(
    "Eligible arrival outcomes:        "
    f"{silver_kpi['arrival_delay_eligible_flights']:,}"
)

unknown_arrivals = (
    silver_kpi["total_flights"]
    - silver_kpi["arrival_delay_eligible_flights"]
)

print(f"Excluded NULL outcomes:            {unknown_arrivals:,}")
print(f"Arrival delay >=15 rate:           {silver_arrival_delay_rate:.4%}")

print()
print("GOLD <-> SILVER KPI RECONCILIATION: PASS")

display(
    spark.createDataFrame(release_pdf)
)

StatementMeta(, a1f909f2-5b97-4107-9d67-41168b0bd666, 5, Finished, Available, Finished, False)

GOLD -> SILVER KPI RECONCILIATION
Silver flight rows:                597,919
Silver distinct flight_key:        597,919

Total flights:                     597,919
Cancelled flights:                 5,402
Cancellation rate:                 0.9035%

Arrival delay >=15 flights:       119,417
Eligible arrival outcomes:        591,206
Excluded NULL outcomes:            6,713
Arrival delay >=15 rate:           20.1989%

GOLD <-> SILVER KPI RECONCILIATION: PASS


SynapseWidget(Synapse.DataFrame, 8191f962-99db-4a80-ace3-ab9ad5b1320f)

In [4]:
import os

EXPORT_DIR = "/lakehouse/default/Files/portfolio/w5_15"

os.makedirs(
    EXPORT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Small public fact sample
# ------------------------------------------------------------

sample_columns = [
    "flight_date",
    "carrier_code",
    "flight_number",
    "origin_airport_code",
    "destination_airport_code",
    "crs_departure_time",
    "cancelled_flag",
    "arrival_delayed_15_flag",
    "weather_match_flag",
    "origin_temperature_2m_c",
    "origin_precipitation_mm",
]

sample_pdf = (
    spark.table(FACT_TABLE)
    .select(*sample_columns)
    .orderBy(
        "flight_date",
        "carrier_code",
        "flight_number",
        "origin_airport_code"
    )
    .limit(25)
    .toPandas()
)

sample_path = (
    f"{EXPORT_DIR}/gold_fact_sample_25.csv"
)

sample_pdf.to_csv(
    sample_path,
    index=False
)


# ------------------------------------------------------------
# KPI reconciliation evidence
# ------------------------------------------------------------

kpi_path = (
    f"{EXPORT_DIR}/gold_kpi_release_results.csv"
)

release_pdf.to_csv(
    kpi_path,
    index=False
)


# ------------------------------------------------------------
# Recruiter-readable release note
# ------------------------------------------------------------

summary = f"""# AirOps 360 - Gold Release v0.1

## Release gate

Same-batch rerun: PASS

- Fact rows: {after['fact_rows']:,}
- Distinct flight_key: {after['fact_distinct_flight_keys']:,}
- Duplicate fact business-key groups: {after['fact_duplicate_groups']:,}
- Daily airport aggregate rows: {after['aggregate_rows']:,}
- Duplicate airport-day groups: {after['aggregate_duplicate_groups']:,}

Publication timestamps are intentionally excluded from the
stable rerun signature because they change on every publication.

## KPI reconciliation

Gold KPIs were independently reconciled to
`lh_airops_silver.slv_flights_weather_enriched`.

- Total Flights: {silver_kpi['total_flights']:,}
- Cancellation Rate: {silver_cancellation_rate:.4%}
  ({silver_kpi['cancelled_flights']:,} / {silver_kpi['total_flights']:,})
- Arrival Delay >=15 Rate: {silver_arrival_delay_rate:.4%}
  ({silver_kpi['arrival_delayed_15_flights']:,} /
  {silver_kpi['arrival_delay_eligible_flights']:,})
- Unknown arrival-delay outcomes excluded: {unknown_arrivals:,}

## Scope

Weather remains an ORD/ATL April 2026 pilot.
NULL weather outside pilot coverage means weather data is unavailable,
not that weather was good.

This release demonstrates rerun safety, business-grain integrity,
and Silver-to-Gold KPI reconciliation.
"""

summary_path = (
    f"{EXPORT_DIR}/GOLD_RELEASE_V0.1.md"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(summary)


print("=" * 72)
print("PORTFOLIO EXPORT COMPLETE")
print("=" * 72)
print(sample_path)
print(kpi_path)
print(summary_path)

display(sample_pdf)

StatementMeta(, a1f909f2-5b97-4107-9d67-41168b0bd666, 6, Finished, Available, Finished, False)

PORTFOLIO EXPORT COMPLETE
/lakehouse/default/Files/portfolio/w5_15/gold_fact_sample_25.csv
/lakehouse/default/Files/portfolio/w5_15/gold_kpi_release_results.csv
/lakehouse/default/Files/portfolio/w5_15/GOLD_RELEASE_V0.1.md


SynapseWidget(Synapse.DataFrame, 489c5ff9-1af8-4bbc-8289-edc4c3a8962b)